In [78]:
import numpy as np
import pandas as pd

def multioutput_error_rates(x1, x2, y1, y2, tol_eps=0.05):
    x1, x2, y1, y2 = map(lambda arr: np.asarray(arr, dtype=float), [x1, x2, y1, y2])
    N = len(x1)
    mul = 100.0  # to percent

    e1 = np.abs(y1 - x1)
    e2 = np.abs(y2 - x2)

    mae1 = e1.mean()
    mae2 = e2.mean()
    rmse1 = np.sqrt(((y1 - x1)**2).mean())
    rmse2 = np.sqrt(((y2 - x2)**2).mean())

    max_per_sample = np.maximum(e1, e2)
    mean_maxerr = max_per_sample.mean()

    mean_l2 = np.mean(np.sqrt(e1**2 + e2**2))
    rmse_multi = np.sqrt(((y1 - x1)**2 + (y2 - x2)**2).mean())

    ter = (max_per_sample > tol_eps).mean()

    results = {
        "MAE1": mae1 * mul,
        "MAE2": mae2 * mul,
        "RMSE1": rmse1 * mul,
        "RMSE2": rmse2 * mul,
        "MeanMaxErr": mean_maxerr * mul,
        "MeanL2": mean_l2 * mul,
        "RMSE_multi": rmse_multi * mul,
        "TER": ter * mul
    }
    return results

In [79]:
# 예시 데이터
x1 = [0.1, 0.4, 0.9]
y1 = [0.12, 0.335, 0.82]

x2 = [0.2, 0.8, 0.3]
y2 = [0.25, 0.75, 0.28]

tol_eps = 0.05
errors = multioutput_error_rates(x1, x2, y1, y2, tol_eps)
print(errors)

{'MAE1': np.float64(5.500000000000002), 'MAE2': np.float64(4.0), 'RMSE1': np.float64(6.062177826491073), 'RMSE2': np.float64(4.242640687119286), 'MeanMaxErr': np.float64(6.500000000000002), 'MeanL2': np.float64(7.277328597266066), 'RMSE_multi': np.float64(7.399324293474374), 'TER': np.float64(66.66666666666666)}


In [28]:
# MAE₁ = mean(AE₁), MAE₂ = mean(AE₂), AE₁ = |y1 − x1|, AE₂ = |y2 − x2|
print("평균 절대 오차 [%]")
print(f"{errors['MAE1']:.2f}, {errors['MAE2']:.2f}")

각 출력 오차율 평균 [%]
3.00, 4.00


In [30]:
# RMSE₁ = sqrt(MSE₁), RMSE₂ = sqrt(MSE₂)
print("평균제곱오차의 제곱근 [%]")
print(f"{errors['RMSE1']:.2f}, {errors['RMSE2']:.2f}")

각 출력 오차율 기하평균 [%]
3.32, 4.24


In [44]:
print("샘플 단위 최악 오차율 평균 (평균) [%]")
print(f"{errors['MeanMaxErr']:.2f}")

샘플 단위 최악 오차율 평균 (평균) [%]
6.50


In [45]:
print("유클리드 거리 평균: 두 오차를 동시에 고려한 지표 [%]")
print(f"{errors['MeanL2']:.2f}")

유클리드 거리 평균: 두 오차를 동시에 고려한 지표 [%]
7.28


In [46]:
print(f"허용 오차 초과율 [%], ε=(|err|>{tol_eps})")
print(f"{errors['MeanMaxErr']:.2f}")

허용 오차 초과율 [%], ε=(|err|>0.05)
6.50


# 1. Similarity tool 개발 2차 버전 결과 (25년 08월 18일 기준)
1차 버전에서 1회 업데이트를 했는데도 여전히 오차가 많아서,  
수동으로 결과를 확인하면서 문법/LLM 정의를 다시 함.

In [87]:
import pandas as pd
import numpy as np

# 1. 기준 데이터 (정답)
in_1 = pd.read_csv("utils/cloud_similarity/output_merged_250710.csv")
in_2 = pd.read_csv("utils/cloud_similarity/output_merged_250715.csv")

x1 = in_1["cloud_similarity_gpt4.1-mini_mg_up"].values
x2 = in_2["cloud_similarity_gpt-4.1-mini_cap_hide"].values

# 2. 모델 출력 데이터
out_1 = pd.read_csv("utils/cloud_similarity/output_merged_250715.csv")
out_2 = pd.read_csv("utils/cloud_similarity/output_merged_250710_cap.csv")

y1 = out_1["cloud_similarity_gpt4.1-mini_gpt_mg_up"].values        # ← 실제 컬럼명 확인 필요
y2 = out_2["cloud_similarity_gpt-4.1_cap"].values   # ← 실제 컬럼명 확인 필요

# 3. NaN 필터링: 같은 행에서 하나라도 NaN이 있으면 제외
mask = ~(np.isnan(x1) | np.isnan(x2) | np.isnan(y1) | np.isnan(y2))
print("Length: ", len(x1), " ", len(x2), " ", len(y1), " ", len(y2))

x1_clean, x2_clean, y1_clean, y2_clean = (
    x1[mask], x2[mask], y1[mask], y2[mask]
)
print("Length after filtering:", len(x1_clean), " ", len(x2_clean), " ", len(y1_clean), " ", len(y2_clean))

# 4. 오차율 계산
tol_eps=0.5
errors = multioutput_error_rates(x1_clean, x2_clean, y1_clean, y2_clean, tol_eps) #tol_eps=0.05
print(errors)


Length:  175   175   175   175
Length after filtering: 171   171   171   171
{'MAE1': np.float64(8.676986584152045), 'MAE2': np.float64(8.905400756783626), 'RMSE1': np.float64(15.477615039549992), 'RMSE2': np.float64(15.006425976061863), 'MeanMaxErr': np.float64(13.647746818070171), 'MeanL2': np.float64(14.848357804003653), 'RMSE_multi': np.float64(21.558046940470472), 'TER': np.float64(23.391812865497073)}


In [93]:
print("평균 절대 오차 [%]") # 모든 샘플이 동일하게 기여, 큰 오차에 민감하지 않은 단점
print(f"{errors['MAE1']:.2f}, {errors['MAE2']:.2f}")
print("평균제곱오차의 제곱근 [%]") # 오차를 제곱 후 평균, 다시 루트 → 큰 오차에 더 민감 (일부 샘플은 ±15% 정도까지 오차가 발생)
print(f"{errors['RMSE1']:.2f}, {errors['RMSE2']:.2f}") 
print("샘플 단위 최악 오차율 평균 (평균) [%]") # 한 샘플에서 최악의 오차가 평균적으로 13.65%
print(f"{errors['MeanMaxErr']:.2f}") 

# 사용할 주요 지표
print("\n유클리드 거리 평균: 두 오차를 동시에 고려한 지표 [%]") # 두 출력 모두 잘 맞춰야 하는 모델 평가에 적합
print(f"{errors['MeanL2']:.2f}") 
print(f"허용 오차 초과율 [%], ε=(|err|>{tol_eps})") # 특정 허용 오차 초과율
print(f"{errors['TER']:.2f}")

평균 절대 오차 [%]
8.68, 8.91
평균제곱오차의 제곱근 [%]
15.48, 15.01
샘플 단위 최악 오차율 평균 (평균) [%]
13.65

유클리드 거리 평균: 두 오차를 동시에 고려한 지표 [%]
14.85
허용 오차 초과율 [%], ε=(|err|>0.3)
23.39


# 2. Similarity tool 개발 3차 버전 결과 (25년 08월 22일 기준)

In [95]:
print("평균 절대 오차 [%]") # 모든 샘플이 동일하게 기여, 큰 오차에 민감하지 않은 단점
print(f"3.28, 4.23")
print("평균제곱오차의 제곱근 [%]") # 오차를 제곱 후 평균, 다시 루트 → 큰 오차에 더 민감 (일부 샘플은 ±15% 정도까지 오차가 발생)
print(f"7.13, 7.08") 
print("샘플 단위 최악 오차율 평균 (평균) [%]") # 한 샘플에서 최악의 오차가 평균적으로 13.65%
print(f"5.82") 

# 사용할 주요 지표
print("\n유클리드 거리 평균: 두 오차를 동시에 고려한 지표 [%]") # 두 출력 모두 잘 맞춰야 하는 모델 평가에 적합
print(f"6.34") 
print(f"허용 오차 초과율 [%], ε=(|err|>{tol_eps})") # 특정 허용 오차 초과율
print(f"12.38")

평균 절대 오차 [%]
3.28, 4.23
평균제곱오차의 제곱근 [%]
7.13, 7.08
샘플 단위 최악 오차율 평균 (평균) [%]
5.82

유클리드 거리 평균: 두 오차를 동시에 고려한 지표 [%]
6.34
허용 오차 초과율 [%], ε=(|err|>0.3)
12.38
